# Mixup en línea (R6) — generar los vectores sintéticos dentro de cada fold y evaluar

Mixup no genera texto: interpola **embeddings** de dos oraciones reales de la misma categoría (`x = λ·xᵢ + (1-λ)·xⱼ`, `λ ~ Beta(0.4, 0.4)`).
Todo lo hace `validacion_cruzada_en_linea.py`.

**Estrategia (la misma que el entrenamiento): test / pool / core / dev.** Para cada fold: *test* = el fold (no se toca);
*pool* = los otros 4 folds; el pool se parte en *dev interno* (~1/6) y *core*. Hay dos etapas:
- **Etapa core (elegir `C`):** el aumento se genera/filtra solo con el core y se mide en el dev interno.
- **Etapa pool (modelo final):** el aumento se genera/filtra con todo el pool y se mide en el test.

Así, el dev interno nunca influye en el aumento con el que se elige `C`. En todas las etapas se descartan las filas sintéticas
idénticas a una oración del test.

En Mixup: la etapa core interpola solo oraciones del core y la etapa pool interpola oraciones del pool (así ya funciona la evaluación vigente). No
necesita API ni GPU. Como los vectores dependen de una semilla, este cuaderno repite la validación cruzada completa con **varias semillas** para medir
cuánto del resultado es azar de la interpolación. La semilla 0 reproduce el resultado vigente: sirve de control.

**Antes de correr:** `git commit` + `git push` del repo reorganizado (carpetas `oe2_...`) con `validacion_cruzada_en_linea.py`.

**Tiempo estimado:** unos 10 minutos por semilla en CPU (casi todo es extraer embeddings); con GPU, menos.

## 1. Clonar tu repo

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3

ruta = '/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py'
assert os.path.exists(ruta), ('No existe validacion_cruzada_en_linea.py: haz git commit + git push del repo reorganizado '
                              '(carpetas oe2_...) con los cambios nuevos y vuelve a correr esta celda.')
print('OK')

## 2. Instalar dependencias

In [ ]:
!pip install -q sentence-transformers scikit-learn
import torch
print('GPU disponible:', torch.cuda.is_available())

## 3. Verificar corpus y folds

In [ ]:
import os, subprocess

def correr(comando, cwd="/content/shiwilu-tesis"):
    """Corre un comando mostrando la salida en vivo; se detiene si falla."""
    proc = subprocess.Popen(
        comando, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"},
    )
    for linea in proc.stdout:
        print(linea, end="")
    proc.wait()
    assert proc.returncode == 0, f"Fallo (codigo {proc.returncode}): {' '.join(comando)}"

verificacion = """
import numpy as np, pandas as pd
from shiwilu.clasificacion import cargar_corpus, cargar_folds
c = cargar_corpus(); f = cargar_folds(c)
print(len(c), 'oraciones; tamano de cada fold:', np.bincount(f).tolist())
print(pd.crosstab(f, c.intencion))
"""
correr(["python", "-c", verificacion])

## 4. Mixup en línea, una corrida por semilla

`SEMILLAS` es editable (la 0 es el control). Cada corrida guarda sus CSV con la etiqueta `colab_s<semilla>`.

In [ ]:
SEMILLAS = [0, 1, 2]

for semilla in SEMILLAS:
    print(f"\n=============== Mixup, semilla de aumento {semilla} ===============")
    correr(["python", "oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py",
            "--tecnica", "mixup", "--semilla-aumento", str(semilla), "--etiqueta", f"colab_s{semilla}"])

## 5. Tablas y curvas ROC (las mismas que el repo), por semilla

El CSV de predicciones de cada corrida trae **los 12 experimentos** (Mixup de esa semilla + las otras tres configuraciones vigentes) con las
probabilidades por categoría. Por cada semilla se generan: `comparacion_pareada_sin_puntuacion_<TAG>.csv`, `curvas_roc_<TAG>.csv`, `curvas_roc_<TAG>.png`
(grilla 3 × 4) y la carpeta `curvas_roc_<TAG>/` con una imagen por experimento (`TAG = en_linea_mixup_colab_s<semilla>`).

In [ ]:
import sys
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion")
from comparacion_pareada import diferencia_pareada

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
vigente = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion.csv")
P_vig = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion.csv")

def f1(tabla, tecnica):
    return tabla[tabla.tecnica == tecnica].set_index("modelo").f1_macro_agrupado

from IPython.display import Image

for semilla in SEMILLAS:
    TAG = f"en_linea_mixup_colab_s{semilla}"
    ENTRADA = f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv"
    correr(["python", "oe2_aumento_de_datos/evaluacion/comparacion_pareada.py", "--entrada", ENTRADA, "--etiqueta", TAG])
    correr(["python", "oe2_aumento_de_datos/evaluacion/curvas_roc.py", "--entrada", ENTRADA, "--etiqueta", TAG])
    display(Image(f"{RES}/curvas_roc_{TAG}.png"))
    auc_macro = pd.read_csv(f"{RES}/curvas_roc_{TAG}.csv").query("categoria == 'macro'")
    display(auc_macro.pivot(index="modelo", columns="tecnica", values="auc").round(4))

## 6. Resumen por semilla

Control: la semilla 0 debe dar (casi) exactamente el F1 de Mixup del resultado vigente.

In [ ]:
f1_sin, f1_mix = f1(vigente, "sin_aumento"), f1(vigente, "mixup")
por_semilla = {}
for semilla in SEMILLAS:
    r = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion_en_linea_mixup_colab_s{semilla}.csv")
    por_semilla[f"mixup s{semilla}"] = f1(r, "mixup")
if 0 in SEMILLAS:
    dif = (por_semilla["mixup s0"] - f1_mix).abs().max()
    print(f"Control: |F1 semilla 0 - F1 Mixup vigente| maximo = {dif:.4f}")
    assert dif < 0.01, "La semilla 0 no reproduce el Mixup vigente: algo cambio"

tabla = pd.DataFrame({"sin aumento (vigente)": f1_sin, **por_semilla})
cols = list(por_semilla)
tabla["Mixup media"] = tabla[cols].mean(axis=1)
tabla["Mixup sd"] = tabla[cols].std(axis=1, ddof=1) if len(cols) > 1 else np.nan
tabla["media - sin aumento"] = tabla["Mixup media"] - tabla["sin aumento (vigente)"]
display(tabla.round(4))

## 7. Diferencia pareada contra `sin_aumento` (IC95% bootstrap), por semilla

In [ ]:
rng = np.random.default_rng(42)
filas = []
for semilla in SEMILLAS:
    P_s = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_en_linea_mixup_colab_s{semilla}.csv")
    for modelo in ["labse", "mbert", "xlmr"]:
        d, lo, hi = diferencia_pareada(P_s, (modelo, "mixup"), (modelo, "sin_aumento"), rng)
        filas.append({"semilla": semilla, "modelo": modelo, "diferencia_f1": round(d, 4),
                      "ic95_bajo": round(lo, 4), "ic95_alto": round(hi, 4), "distinguible_de_cero": not (lo <= 0 <= hi)})
display(pd.DataFrame(filas))

## 8. Descargar el resultado

Un `.zip` con los resultados de cada semilla: predicciones y resumen (12 experimentos), comparaciones pareadas y curvas ROC (CSV, grilla e imágenes por experimento).
En tu computadora van en `oe2_aumento_de_datos/evaluacion/resultados/`.

In [ ]:
TAG = "en_linea_mixup_colab_s"

import glob, shutil
from google.colab import files

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
carpeta = "/content/salida_mixup_en_linea"
shutil.rmtree(carpeta, ignore_errors=True)
os.makedirs(carpeta + "/resultados")
for ruta_ in glob.glob(f"{RES}/*{TAG}*"):      # CSV, PNG y la carpeta con una imagen por experimento
    destino = carpeta + "/resultados/" + os.path.basename(ruta_)
    shutil.copytree(ruta_, destino) if os.path.isdir(ruta_) else shutil.copy(ruta_, destino)
zip_path = shutil.make_archive("/content/salida_mixup_en_linea", "zip", carpeta)
print(sorted(os.listdir(carpeta + "/resultados")))
files.download(zip_path)